# Ingeniero de boxes para simracing

**Carlos Peñalver · Proyecto del módulo de IA Generativa**

Estado: corpus indexado con Gemini Embedding 2; recuperación inicial comprobada. Agente y memoria pendientes. Las celdas de preparación funcionan sin llamadas a Gemini. El agente todavía no está implementado; las secciones pendientes no constituyen una demo funcional.

Stack: Gemini (LLM y embeddings), ChromaDB, LangGraph y LangChain. Interfaz Streamlit como bonus después del núcleo.

Plan: `../docs/PLAN.md`. Fuentes: `../docs/FUENTES.md`. Evaluación: `../docs/EVALUACION.md`.


## 0. Preparación del entorno

Seleccionar el Python de `.venv` de **Proyecto**, no el entorno de los notebooks de clase. Instalar antes `requirements.txt` siguiendo el README.

Esta celda localiza la raíz desde `Proyecto/` o `Proyecto/notebooks/`. Solo carga el `.env` de este proyecto y no muestra las claves. El .env de Proyecto tiene prioridad sobre variables heredadas del sistema. El modelo de embeddings es el mismo de clase; el LLM se elegirá en la siguiente fase.


In [1]:
from pathlib import Path
from importlib.metadata import version
import os
import sys

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "requirements.txt").is_file() or not (ROOT / "prompts/system.md").is_file():
    raise RuntimeError("Abre el notebook desde Proyecto o Proyecto/notebooks.")

load_dotenv(ROOT / ".env", override=True)
CORPUS_DIR = ROOT / "data/corpus"
CHROMA_DIR = ROOT / "data/chroma"
SYSTEM_PROMPT = (ROOT / "prompts/system.md").read_text(encoding="utf-8")

print("Python:", sys.version.split()[0])
print("Proyecto:", ROOT.name)
for package in ("langchain-google-genai", "langgraph", "chromadb", "pypdf"):
    print(f"{package}: {version(package)}")
for name in ("GOOGLE_API_KEY", "GEMINI_MODEL", "GEMINI_EMBEDDING_MODEL"):
    print(f"{name}: {'configurado' if os.getenv(name, '').strip() else 'pendiente'}")


Python: 3.12.3
Proyecto: Proyecto
langchain-google-genai: 4.4.2
langgraph: 1.2.15
chromadb: 1.5.9
pypdf: 6.20.1
GOOGLE_API_KEY: configurado
GEMINI_MODEL: pendiente
GEMINI_EMBEDDING_MODEL: configurado


In [2]:
# Comprobación de importaciones: no instancia clientes ni realiza llamadas de API.
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from pypdf import PdfReader

print("Importaciones del stack correctas. No se ha llamado a Gemini.")


Importaciones del stack correctas. No se ha llamado a Gemini.


## 1. Dominio y corpus — preparado

Ingeniero de boxes para iRacing, ACC y LMU. El alumno ha elegido tres fuentes: árbol público de Ultimate Setup Hub (25 diagnósticos), chuleta de Chris Haye en SimRacingCockpit (20 fichas) y Driver61 ACC Setup Guide V1 (30 fichas).

Son **tres fuentes y 75 fichas**, no 75 documentos originales ni 20 páginas artificiales. Inventario activo: `docs/corpus_mvp.json`. Procedencia, preparación y reproducción: `docs/FUENTES.md`. Los originales, transcripciones y JSONL son locales y no están en Git; las seis fuentes anteriores quedan en reserva.

Se revisó la transcripción de las láminas visualmente; no se validaron los ajustes en pista ni su vigencia para todos los coches. La evaluación medirá fidelidad documental, citas y memoria, sin necesitar simulador.


## 2. Carga, embeddings e índice Chroma

La celda siguiente verifica el inventario y carga las 75 fichas como `Document`, con fuente y simulador. Cada caso completo es un fragmento: conserva síntoma, fase, velocidad y alternativas. Por eso no volvemos a cortar por caracteres las celdas ya preparadas.

Seguimos `rag_profesor.ipynb` (celdas 40–45) y `clase3.ipynb` (35–40): `GoogleGenerativeAIEmbeddings`, `Chroma.from_documents`, persistencia y `similarity_search`. La adaptación compartida está en `retrieval.py`: añade compatibilidad con Embedding 2, reutilización del índice y filtro obligatorio por simulador.

Las celdas de carga no llaman a Gemini. La creación del índice sí envía el corpus a Gemini para generar embeddings; las búsquedas también requieren un embedding de cada consulta. Repetir la indexación del mismo corpus reutiliza lo guardado. No se generan todavía respuestas de chat.


In [3]:
from collections import Counter
import runpy
from langchain_core.documents import Document

builder = runpy.run_path(str(ROOT / "scripts/build_corpus.py"))
records = builder["build"](ROOT)
documents = [
    Document(page_content=record["text"], metadata={
        key: value for key, value in record.items() if key != "text"
    })
    for record in records
]
counts = Counter(doc.metadata["simulator"] for doc in documents)
assert counts == {"lmu": 25, "iracing": 20, "acc": 30}
assert len({doc.metadata["id"] for doc in documents}) == 75
print("Fuentes:", len({doc.metadata["source_id"] for doc in documents}))
print("Fichas por simulador:", dict(counts))
print("Fragmentos completos:", len(documents))
print("Carga local completada; esta celda no llama a Gemini.")


Fuentes: 3
Fichas por simulador: {'lmu': 25, 'iracing': 20, 'acc': 30}
Fragmentos completos: 75
Carga local completada; esta celda no llama a Gemini.


### 2.1. Modelo de embeddings de clase

Usamos `gemini-embedding-2`, sin reducir sus 3072 dimensiones. El cliente hereda de `GoogleGenerativeAIEmbeddings` y conserva su procesamiento por lotes.

La versión instalada de la integración (4.4.2) envía `task_type` por defecto. Embedding 2 requiere omitirlo y usar prefijos de búsqueda en el texto; lo resuelve `GeminiRetrievalEmbeddings`, manteniendo el contenido original en Chroma. [Documentación de Google](https://ai.google.dev/gemini-api/docs/embeddings#task-types) y [tarifas/cuotas del nivel gratuito](https://ai.google.dev/gemini-api/docs/pricing#gemini-embedding-2). No se activa facturación ni se cambian cuotas.


In [4]:
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from retrieval import GeminiRetrievalEmbeddings, index_documents, search

MODEL_EMBEDDING = os.getenv("GEMINI_EMBEDDING_MODEL", "").strip()
if MODEL_EMBEDDING != "gemini-embedding-2":
    raise ValueError("Configura GEMINI_EMBEDDING_MODEL=gemini-embedding-2 en Proyecto/.env.")
if not os.getenv("GOOGLE_API_KEY", "").strip():
    raise ValueError("Falta GOOGLE_API_KEY en Proyecto/.env.")

llm_embedding = GeminiRetrievalEmbeddings(
    model=MODEL_EMBEDDING,
    api_key=os.environ["GOOGLE_API_KEY"],
    vertexai=False,
)
print("Embeddings:", MODEL_EMBEDDING, "— 3072 dimensiones")


Embeddings: gemini-embedding-2 — 3072 dimensiones


### 2.2. Crear o abrir Chroma

`index_documents` utiliza `Chroma.from_documents`, como en clase, para las fichas que aún no están guardadas. Los ID son estables. El nombre de colección depende del corpus, modelo y formato de embeddings: así no se mezclan índices incompatibles. Si cambian, se crea otra colección; la anterior se conserva.

Chroma persiste en `data/chroma/`; esta carpeta y las credenciales están excluidas de Git. La primera ejecución genera los embeddings del corpus; las siguientes lo reutilizan.


In [5]:
vectorstore = index_documents(documents, llm_embedding, CHROMA_DIR)
stored_ids = vectorstore.get(include=[])["ids"]
assert set(stored_ids) == {doc.metadata["id"] for doc in documents}
print("Fichas en Chroma:", len(stored_ids))
print("Colección:", vectorstore._collection.name)

reopened = index_documents(documents, llm_embedding, CHROMA_DIR)
assert set(reopened.get(include=[])["ids"]) == set(stored_ids)
print("Índice reabierto: mismas 75 fichas, sin duplicados ni nuevos embeddings del corpus.")


Fichas en Chroma: 75
Colección: simracing_1809b2209d21ea5366b5
Índice reabierto: mismas 75 fichas, sin duplicados ni nuevos embeddings del corpus.


### 2.3. Comprobar búsquedas antes del agente

Como en clase, hacemos búsqueda por similitud con `k=3`. `search` aplica `filter={"simulator": simulador}` a `similarity_search` y rechaza un simulador desconocido. La consulta sigue en español, aunque las guías están en inglés.

Ejecutaremos solo tres consultas, una por simulador. Cada una tiene una ficha esperada identificada en el corpus. Comprobamos que aparece entre los tres resultados y que todos pertenecen al simulador elegido. Es una prueba inicial de recuperación, no una medida completa de calidad ni una validación de reglajes en pista. Volver a ejecutar esta celda consume tres embeddings de consulta.


In [6]:
from datetime import datetime, timezone
import hashlib
import json

queries = [
    ("lmu", "En LMU, tengo sobreviraje al acelerar a la salida de una curva lenta. ¿Qué propone la guía?", "LMU_USH_24"),
    ("iracing", "En iRacing bloqueo a menudo los neumáticos delanteros al frenar. ¿Qué reparto de frenada sugiere la chuleta?", "IR_CHRIS_HAYE_02"),
    ("acc", "En ACC el coche resulta incómodo e inestable al pasar por los pianos. ¿Qué ajustes indica la guía?", "ACC_DRIVER61_18"),
]
retrieval_results = []
for simulator, query, expected_id in queries:
    results = search(vectorstore, query, simulator, k=3)
    isolated = bool(results) and all(doc.metadata["simulator"] == simulator for doc in results)
    found = expected_id in {doc.metadata["id"] for doc in results}
    retrieval_results.append({
        "simulator": simulator, "query": query, "expected_id": expected_id,
        "expected_in_top3": found, "correct_simulator": isolated,
        "hits": [{key: doc.metadata[key] for key in ("id", "source_id", "title", "locator", "url")}
                 for doc in results],
    })
    print(f"{simulator}: ficha esperada en top 3 = {found}; filtro correcto = {isolated}")
    for doc in results:
        print(" ", doc.metadata["id"], "—", doc.metadata["title"])

report = {
    "executed_at": datetime.now(timezone.utc).isoformat(),
    "embedding_model": MODEL_EMBEDDING, "dimensions": 3072, "k": 3,
    "corpus_sha256": hashlib.sha256((ROOT / "data/corpus/mvp/preparado/corpus.jsonl").read_bytes()).hexdigest(),
    "collection": vectorstore._collection.name,
    "results": retrieval_results,
}
report_path = ROOT / "outputs/retrieval_evaluation.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
assert all(result["correct_simulator"] for result in retrieval_results), "Se mezclaron simuladores."
assert all(result["expected_in_top3"] for result in retrieval_results), "Revisar la recuperación antes del agente."
print("Resultado local guardado en outputs/retrieval_evaluation.json")


lmu: ficha esperada en top 3 = True; filtro correcto = True
  LMU_USH_24 — Oversteer / Corner Exit / Low Speed
  LMU_USH_22 — Oversteer / Middle of Corner / Low Speed
  LMU_USH_18 — Understeer / Corner Exit / Low Speed
iracing: ficha esperada en top 3 = True; filtro correcto = True
  IR_CHRIS_HAYE_20 — Tips / Brake bias
  IR_CHRIS_HAYE_02 — Braking / Front tyres often lock
  IR_CHRIS_HAYE_05 — Braking / Excessive locking
acc: ficha esperada en top 3 = True; filtro correcto = True
  ACC_DRIVER61_23 — Other / Generally unstable
  ACC_DRIVER61_18 — Tyres / Uncomfortable taking kerbs
  ACC_DRIVER61_08 — Corners / Mid-corner / Unstable
Resultado local guardado en outputs/retrieval_evaluation.json


## 3. System prompt — borrador disponible

El borrador está en `prompts/system.md` y se carga en `SYSTEM_PROMPT`. Su justificación está en el README. Ajustarlo al corpus real antes de usarlo.

Debe delimitar dominio, tono, fuentes, aplicabilidad por simulador/coche/versión, falta de información y tratamiento del historial. Distinguir diagnóstico provisional, prueba propuesta y resultado observado. Mostrar en la entrega el texto final utilizado, sin secretos.


## 4. Agente en LangGraph — pendiente

Implementar recepción de pregunta, recuperación de Chroma y generación con Gemini. Recuperar contexto en **cada consulta** y entregar identificadores de fuente al modelo.

Compartir la lógica con la futura interfaz web. Dibujar o mostrar el grafo final y explicar sus nodos. No afirmar que un reglaje se ha aplicado o ha mejorado el coche si el piloto no lo ha confirmado. Un solo grafo utilizará el contexto de simulador para filtrar la recuperación.


## 5. Memoria — pendiente

Añadir un checkpointer y un identificador de conversación. Usar el historial para interpretar preguntas de seguimiento antes de recuperar documentos.

Demostrar una referencia a una respuesta anterior y comprobar que una conversación nueva no hereda mensajes de otra. Diferenciar cambios recomendados y aplicados; al cambiar de simulador o coche, revisar el contexto y no arrastrar valores anteriores. Documentar que la memoria inicial en RAM se pierde al reiniciar.


## 6. Chat y cinco ejemplos documentados — pendiente

Crear una celda de chat interactivo con una salida clara, por ejemplo `/salir`. Después, registrar al menos cinco preguntas y sus respuestas **reales**, con fuentes y un seguimiento. No sustituir las respuestas por expectativas.

| Caso | Pregunta prevista | Respuesta real y fuentes |
| --- | --- | --- |
| E01 | Diagnóstico inicial sin contexto suficiente | Pendiente de ejecución |
| E02 | iRacing: síntoma y prueba de ajuste documentada | Pendiente de ejecución |
| E03 | ACC: consulta documental sobre pianos | Pendiente de ejecución |
| E04 | LMU: sobreviraje por fase y velocidad | Pendiente de ejecución |
| E05 | Seguimiento de E02: cambio aplicado y nuevo síntoma | Pendiente de ejecución |

Usar el corpus elegido y los casos de docs/EVALUACION.md. Identificar los escenarios ficticios del piloto y añadir casos sin evidencia, setup fijo y cambio de simulador. Las respuestas de Gemini se documentarán tras ejecutarlas; no equivalen a validación en pista.


## 7. Comparación con Gemini sin RAG — pendiente

Seguir `docs/EVALUACION.md`: mismo modelo, configuración y preguntas; guardar respuestas reales de ambas condiciones. Dar a ambas variantes el mismo contexto e historial del piloto; los fragmentos recuperados son la diferencia. Separar conceptos generales y consultas específicas de cada simulador.

Valorar corrección, aplicabilidad, memoria, citas y utilidad. Registrar fallos y límites; no asumir que RAG gana ni que una respuesta demuestra mejoras de tiempo. Si hay tiempo, comparar también pegando el mismo material documental a Gemini.


## 8. Bonus: Streamlit y presentación — pendiente

Crear una interfaz de chat sobre el mismo núcleo, con selector de iRacing/ACC/LMU, contexto de coche y condiciones, historial aislado por sesión y fuentes visibles. Añadir Streamlit a las dependencias cuando se implemente y verificar secretos antes de desplegar.

Preparar demo de 5–10 minutos y comprobar la ejecución del notebook desde un kernel limpio. Revisar que sus salidas no contienen credenciales ni información privada.
